# Chapter 8 — Human in the Loop

## Setup Instructions

To ensure you have the required dependencies to run this notebook, you'll need to have our `llm-agents-from-scratch` framework installed on the running Jupyter kernel. To do this, you can launch this notebook with the following command while within the project's root directory:

```sh
uv run --with jupyter jupyter lab
```

Alternatively, if you just want to use the published version of `llm-agents-from-scratch` without local development, you can install it from PyPi by uncommenting the cell below.

In [ ]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPi
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code provided in this notebook, you'll need to have Ollama installed on your local machine and have its LLM hosting service running. To download Ollama, follow the instructions found on this page: https://ollama.com/download. After downloading and installing Ollama, you can start a service by opening a terminal and running the command `ollama serve`.

In [1]:
import os, shutil, subprocess, time, urllib.request, urllib.error


def ensure_ollama(host="http://localhost:11434", timeout=15):
    """Start Ollama if not already running and wait until responsive."""

    def _up():
        try:
            urllib.request.urlopen(f"{host}/api/tags", timeout=1)
            return True
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            return False

    if _up():
        return print(f"✓ Ollama already running at {host}")

    # Lightning persistent path first, then standard locations
    ollama_path = shutil.which("ollama")
    if ollama_path is None:
        for candidate in [
            "/teamspace/studios/this_studio/.local/bin/ollama",
            "/usr/local/bin/ollama",
            "/usr/bin/ollama",
        ]:
            if os.path.exists(candidate):
                ollama_path = candidate
                break
    if ollama_path is None:
        raise RuntimeError(
            "Could not find the ollama binary. Install with: "
            "curl -fsSL https://ollama.com/install.sh | sh"
        )

    print(f"Starting Ollama server ({ollama_path})...")
    subprocess.Popen(
        [ollama_path, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    deadline = time.time() + timeout
    while time.time() < deadline:
        if _up():
            return print(f"✓ Ollama up and running at {host}")
        time.sleep(0.5)

    raise RuntimeError(f"Ollama did not start within {timeout}s")


ensure_ollama()

✓ Ollama already running at http://localhost:11434


## Examples

### Example 1: Agent-Initiated Pause with HumanInputTool

In [2]:
import logging

from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.logger import enable_console_logging
from llm_agents_from_scratch.llms import OllamaLLM
from llm_agents_from_scratch.tools import SimpleFunctionTool
from llm_agents_from_scratch.tools.default import HumanInputTool

enable_console_logging(logging.INFO)


def next_number(x: int) -> int:
    if x % 2 == 0:
        return x // 2
    return 3 * x + 1


next_number_tool = SimpleFunctionTool(func=next_number)
human_input_tool = HumanInputTool()
llm = OllamaLLM(model="qwen3:14b", think=False)
agent = LLMAgent(llm=llm, tools=[next_number_tool, human_input_tool])

In [3]:
task = Task(
    instruction=(
        "Compute the full Hailstone sequence step by step using next_number. "
        "The starting number must come from the human operator — ask them "
        "before beginning."
    )
)
result = await agent.run(task)
print(result.content)

INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: Compute the full Hailstone sequence step by step using next_number. The starting number must come from the human operator — ask them ...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Compute the full Hailstone sequence step by step using next_number. The starting number must come from the human operator — ask th...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      ✅ Step Result: I need to ask the human operator for the starting number to compute the Hailstone sequence. Let me prompt them for it.
INFO (llm_agents_fs.TaskHandler) :      🧠 New Step: Prompt the human operator to provide the starting number for the Hailstone sequence.
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Prompt the human operator to provide the starting number for the Hailstone sequence.
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: human_input


Please provide the starting number for the Hailstone sequence. 4


INFO (llm_agents_fs.TaskHandler) :      ✅ Successful Tool Call: 4
INFO (llm_agents_fs.TaskHandler) :      ✅ Step Result: I have received the starting number, which is 4. Now I will compute the Hailstone sequence step by step using the next_number tool unti...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      🧠 New Step: Compute the next number in the Hailstone sequence using the next_number tool starting from 4.
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Compute the next number in the Hailstone sequence using the next_number tool starting from 4.
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: next_number
INFO (llm_agents_fs.TaskHandler) :      ✅ Successful Tool Call: 2
INFO (llm_agents_fs.TaskHandler) :      ✅ Step Result: The next number in the Hailstone sequence after 4 is 2. I will now compute the next number in the sequence using the next_number tool. ...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      🧠 New Step: Compute the next number in the